# Vectorized vs Normal

In [1]:
import random
from utils import matmul
from modules.autograd import Node


random.seed(42)
in_features = 2
out_features = 1
x = [
    [1.1, 2.2],
    [3.3, 4.4]
]

weights = [
    [Node(random.uniform(-1, 1))
        for _ in range(out_features)]
    for _ in range(in_features)
]
biases = [
    Node(random.uniform(-1, 1))
    for _ in range(out_features)
]

z = matmul(x, weights)
# Add biases
mat_mul_ouptut = [
    [
        z[i][j] + biases[j]
        for j in range(out_features)
    ]
    for i in range(len(z))
]

In [2]:
from modules.modules import Neuron as neuron_cls

neurons = [neuron_cls(n=in_features) for _ in range(out_features)]
output = [n(sample) for sample in x for n in neurons]

In [3]:
output, mat_mul_ouptut

([Node(-2.2331550836746827, ops=+, grad=0),
  Node(-3.70962984748026, ops=+, grad=0)],
 [[Node(-2.2331550836746827, ops=+, grad=0)],
  [Node(-3.70962984748026, ops=+, grad=0)]])

In [ ]:
from autograd import Node

# prev layer ouptut
x1 = Node(data=2, label='x1')
w11 = Node(data=0.1, label='w11')
x2 = Node(data=2, label='x2')
w12 = Node(data=0.1, label='w12')

w21 = Node(data=0.66, label='w21')
w22 = Node(data=0.01, label='w22')

# current layer forward
n1_act = sum((xi*wi for xi, wi in zip([x1, x2], [w11, w12])), start=0.3)
n2_act = sum((xi*wi for xi, wi in zip([x1, x2], [w21, w22])), start=0.64)

n1_act.data, n2_act.data

In [ ]:
import numpy as np
# prev layer ouptut
X = np.array([[x1, x2]])  # (1, 2)

w = np.array([
    [w11, w21],
    [w12, w22]
])  # (2, 2)

b = np.array([[0.3, 0.64]])  # (1, 2)

X @ w + b  # (1, 2)

In [ ]:
from typing import LiteralString, Any
import random
import numpy as np
from autograd import Node

def matmul(A, B):

    rows_A = len(A)
    cols_A = len(A[0])

    rows_B = len(B)
    cols_B = len(B[0])

    assert cols_A == rows_B, "Incompatible matrix dimensions"

    return [
        [
            sum(
                (A[i][k] * B[k][j] for k in range(cols_A)),
                start=Node(0)
            )
            for j in range(cols_B)
        ]
        for i in range(rows_A)
    ]

class Neuron:

    def __init__(self, n: int, weights: list | None = None, bias: float | None = None) -> None:
        self.weights = weights if weights else [Node(data=random.uniform(-1, 1)) for _ in range(n)]
        self.bias = bias if bias is not None else Node(random.uniform(-1, 1))

    def __call__(
            self, 
            X: list | np.ndarray, # previous layers neurons activation values
            activation_fn: str | None = 'relu') -> Node | Any | float:
        #TODO: convert to matric mul for optimization
        activation = sum((xi*wi for xi, wi in zip(X, self.weights)), start=self.bias)
        if activation_fn == 'relu':
            activation = activation.relu()
        return activation

    def parameters(self)-> list | list[Node]:
        return self.weights + [self.bias]

class LinearLayer:

    def __init__(self, in_features: int, out_features: int):
        self.in_features = in_features
        self.out_features = out_features
        self.weights = [
            [Node(random.uniform(-1, 1))
             for _ in range(out_features)]
            for _ in range(in_features)
        ]
        self.biases = [
            Node(random.uniform(-1, 1))
            for _ in range(out_features)
        ]

    def __call__(self, x):
        # Convert input to a 2D matrix if necessary
        if not isinstance(x[0], list): x = [x]

        z = matmul(x, self.weights)

        # Add biases
        return [
            [
                z[i][j] + self.biases[j]
                for j in range(self.out_features)
            ]
            for i in range(len(z))
        ]

    def parameters(self):
        return (
            [w for row in self.weights for w in row]
            + self.biases
        )

class Network:

    def __init__(self, layers: list):
        self.layers = [
            LinearLayer(layers[i], layers[i+1])
            for i in range(len(layers) - 1)
        ]

    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            # ReLU on hidden layers
            if i != len(self.layers) - 1:
                x = [
                    [node.relu() for node in row]
                    for row in x
                ]
        return x

    def parameters(self):
        return [
            param
            for layer in self.layers
            for param in layer.parameters()
        ]

    def zero_grad(self):
        for param in self.parameters():
            param.grad = 0

In [ ]:
# Generate linear dataset
import matplotlib.pyplot as plt

X = np.linspace(-5, 5, 30)
def linear(X_axis: float) -> float: 
    return (X_axis*2.3 + 2) + random.uniform(-5, 5)

def parabola(X_axis: float) -> float:
    return (X_axis**2)# + random.uniform(-5, 5)

def quadratic_porabola(X_axis: float) -> float:
    return (X_axis**4 - 5*X_axis**2 + 4)# + random.uniform(-5, 5)

y = [linear(x_val) for x_val in X]

# pyplot.plot(X_axis, y)
plt.scatter(X, y)

In [ ]:
# regression example
mass = [[x] for x in X]
epoch_predictions = []
net = Network(layers=[1, 4, 1])
lr = 0.0005
# Training
for epoch in range(100):
    # Forward pass
    y_hats = [net(x) for x in mass]
    epoch_predictions.append([y_hat.data for y_hat in y_hats])
    # Calculate loss (SSE)
    # loss = sum(
    #     (y_hat - y)**2
    #     for y_hat, y in zip(y_hats, ys)
    # )
    # Calculate loss (SSE)
    #TODO: Add all operations to the value class to use this loss fn
    loss = sum(
        (y_hat - y)**2
        for y_hat, y in zip(y_hats, y)
    ) / (2 * len(y))

    net.zero_grad()
    # Backpropagation
    loss.backward()

    # Gradient descent
    for param in net.parameters():
        param.data -= lr * param.grad

    print(f"Epoch {epoch}: Loss = {loss.data:.6f}")